# Markets of Whiterun (Medieval Village) Agent-Based Simulation Models

Have you ever noticed how the towns and cities in fantasy and historical fiction never seem to have enough farms?

The goal of this project is to build an agent-based simulation model with decisions made by Cobb-Douglas marginal utility to find the number and behavior of farmers that lead to stable populations and stable commodity prices. We begin by running experiments meant to both find the range of parameters that lead to successful simulations, and establish the functionality of progressively more complex market mechanisms.
Once we have established a range of preference parameters that lead to stability, we will be able to derive world parameters from media and assess the resulting market dynamics--thus the titular markets of Whiterun.

The bulk of the code for this project will be done in the `.py` files, while results and summaries are saved for the `.ipynb` files.


In [1]:
# === IMPORT SHARED FUNCTIONS === #
import whiterunMarket as wrm  # import shared functions
import inspect                # allow us to see function source code in console


## World Parameters

The power of agent-based simulation models rests in its parameters. The terms listed below determine the basic rules by which our agents act.

For the sake of information gathering, we specify that each experiment runs 1,000 trials, `VILLAGES = int(1E3)`. Since our preference parameters are drawn from a random distribution, we need to ensure a large enough net is cast in each experiment. Each trial is run for 1,000 cycles, `DAYS = int(1E3)`, so as to allow for enough time to pass that . Because our agents don't interact with each other or with a dynamic market in the early experiments, the initial number of `FARMERS` in each trial is largely inconsequential; however, it will become one of the key parameters-of-interest for later experiments.

For the early experiments, the market and production parameters are somewhat arbitrary with the simplifying assumption that any solution produced within a reasonable range of values is congruent to a solution at a larger scale. Moreover, that once a baseline is established, specific scenarios can be assessed. As we develop the market mechanisms through iterative experiments, `PR_BREAD` and `PR_WHEAT` will become initial values for what will be dynamic, market-driven variables. Finally, once we have found that a stable market can be created, we can derive values for parameters like `FARMERS`, `PROD_WHEAT`, `EXHAUSTION`, etc. from fiction or real-world scenarios, and assess the results.


In [2]:
print(
''' # Copied from whiterunMarket.py
# ['EXPERIMENT PARAMS']
VILLAGES   = int(1E3) # number of villages to text
DAYS       = int(1E3) # number of days to simulate

# ['WORLD PARAMS']
FARMERS    = 12  # number of farmer per village
PROD_WHEAT = 3 # amount of wheat produced per production action

# ['MARKET PARAMS']
PR_BREAD   = 3 # initial price of bread
PR_WHEAT   = 1 # initial price of wheat
''')

 # Copied from whiterunMarket.py
# ['EXPERIMENT PARAMS']
VILLAGES   = int(1E3) # number of villages to text
DAYS       = int(1E3) # number of days to simulate

# ['WORLD PARAMS']
FARMERS    = 12  # number of farmer per village
PROD_WHEAT = 3 # amount of wheat produced per production action

# ['MARKET PARAMS']
PR_BREAD   = 3 # initial price of bread
PR_WHEAT   = 1 # initial price of wheat



## Meet the Farmers

We define our farmer agent, or `agentFarmer(...)`, as having an inventory (`gold`, `wheat` and `bread`), a vector of preference params (`PREFS`), statuses (`hunger` and the living or dead binary `status`) and a utility function (the class method `.utility(...)`). Here we deploy the logarithmic-version of the Cobb-Douglas utility function for interpretability and to simulate diminishing returns of the resources.

Utility Function:
$U(G, W, B, H) = \gamma \ln(G+1) + \omega \ln(W+1) + \beta \ln(B+1) - \eta H$

Note that because none of our actions effect only one variable, taking the gradient will not be sufficient. Instead, we take the linear approach:
$\Delta U |_{\vec{x}} = U(X + \vec{x}) - U(X)$ where $X$ is the agent's current inventory and $\vec{x}$ is some change to the inventory.

Since we are taking a largely unsupervised (that is, little or no observed data) approach, the preference params will be drawn from some non-negative random distribution. The extent to which a set of preferences lead to a successful simulation will determine their explanatory power for our purposes--noting that falsifiability comes from the discussion of 

For now, we suppose a small number of Farmer Actions:
* Buy Bread : trades gold for bread
* Produce Wheat : creates wheat at the cost of additional hunger
* Sell Wheat : trades wheat for gold
* Idle : No change (marginal utility hardcoded to `0`)
  
In general, an agent will decide to take the action with the highest marginal utility ($\Delta U |_{\vec{x}}$). An agent cannot do an action they do not have the resources for (i.e. an agent cannot attempt to buy bread if the agent cannot afford bread). If all available actions produce negative marginal utility, then the agent will choose the `Idle` action. A $+1$ is included inside all logarithmic terms so that a `0` of that resource translates to `0` utility, instead of $-\infty$. If an agent's `hunger` exceeds some `STARVATION` threshold, then their `status` is set to `0`. This can reflect a farmer perishing, or could be adjust to also reflect bankruptcy. At the end of the day/ time-step, a farmer will consumer one bread if it is in inventory, reducing `hunger` to `0` or by some `amount`.


In [3]:
print(
    ''' # Copied from whiterunMarket.py
    # ['FARMER PARAMS']
    # Farmer status change parameters
    EXHAUSTION = 1 # amount of hunger gained per production action
    STARVATION = 5 # max amount of hunger allowed before starvation
    RECOVERY   = 0 # eating resets hunger if 0, else reduces hunger
    
    # Initial conditions for farmer agent inventories
    BREAD      = 1
    WHEAT      = 0
    GOLD       = 3
    HUNGER     = 0''',
    '',
    '# Use named tuple to make references easier',
    "PARAMS = ['gamma', 'omega', 'beta', 'eta']",
    "PREFS  = namedtuple('PREFS', PARAMS)",
    '',
    '# Define a class for Farmer',
    inspect.getsource(wrm.agentFarmer),
    sep='\n'
)

 # Copied from whiterunMarket.py
    # ['FARMER PARAMS']
    # Farmer status change parameters
    EXHAUSTION = 1 # amount of hunger gained per production action
    STARVATION = 5 # max amount of hunger allowed before starvation
    RECOVERY   = 0 # eating resets hunger if 0, else reduces hunger

    # Initial conditions for farmer agent inventories
    BREAD      = 1
    WHEAT      = 0
    GOLD       = 3
    HUNGER     = 0

# Use named tuple to make references easier
PARAMS = ['gamma', 'omega', 'beta', 'eta']
PREFS  = namedtuple('PREFS', PARAMS)

# Define a class for Farmer
class agentFarmer :
    def __init__(self, func, **kwargs) :
        self.status = 1
        self.prefs  = PREFS(*func())
        self.bread  = kwargs.get('bread', BREAD)   # initial bread inventory
        self.wheat  = kwargs.get('wheat', WHEAT)   # initial wheat inventory
        self.gold   = kwargs.get('gold',  GOLD)    # initial gold inventory
        self.hunger = kwargs.get('hunger',HUNGER)  # initial hung

... to be continued ...